# 🔧 Feature Engineering — Energy Grid Outage Prediction

Aggregates Silver grid sensor readings per substation per day and joins with
outage events to create features for outage prediction.

**Reads from:** `silver_grid_sensors`, `silver_power_events`

**Writes to:** `gold_ml_features`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, stddev, count,
    sum as spark_sum, max as spark_max, min as spark_min,
    dayofweek, month, abs as spark_abs
)

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load Silver tables (created by batch/02_silver_transform)
sensors_df = spark.read.table('silver_grid_sensors')
events_df = spark.read.table('silver_power_events')

print(f'Silver grid sensors: {sensors_df.count():,} rows')
print(f'Silver power events: {events_df.count():,} rows')

In [ ]:
# === Sensor features: aggregate per substation per day ===
sensor_daily = (
    sensors_df
    .groupBy('substation_id', 'region', 'sensor_date')
    .agg(
        avg('voltage_v').alias('avg_voltage'),
        stddev('voltage_v').alias('std_voltage'),
        spark_min('voltage_v').alias('min_voltage'),
        spark_max('voltage_v').alias('max_voltage'),
        avg('frequency_hz').alias('avg_frequency'),
        stddev('frequency_hz').alias('std_frequency'),
        avg('power_factor').alias('avg_power_factor'),
        spark_min('power_factor').alias('min_power_factor'),
        avg('load_mw').alias('avg_load'),
        spark_max('load_mw').alias('max_load'),
        avg('temperature_c').alias('avg_temp'),
        spark_max('temperature_c').alias('max_temp'),
        count('*').alias('reading_count'),
        spark_sum(when(col('voltage_anomaly'), 1).otherwise(0)).alias('voltage_anomaly_count'),
    )
    .withColumn('voltage_range', col('max_voltage') - col('min_voltage'))
    .withColumn('voltage_deviation', spark_abs(col('avg_voltage') - 230.0))
    .withColumn('freq_deviation', spark_abs(col('avg_frequency') - 50.0))
    .withColumn('anomaly_ratio', col('voltage_anomaly_count') / col('reading_count'))
)

print(f'Sensor daily features: {sensor_daily.count():,} rows')

In [ ]:
# === Outage target: did this substation have a failure event on this day? ===
# Failure-class events (driven by grid stress). 'restoration' is the recovery
# event that follows a failure and is intentionally excluded from the label.
FAILURE_EVENT_TYPES = ['outage', 'voltage_sag', 'surge', 'overload', 'equipment_fault']
outage_events = (
    events_df
    .filter(col('event_type').isin(FAILURE_EVENT_TYPES))
    .groupBy('substation_id', 'event_date')
    .agg(
        count('*').alias('event_count'),
        spark_sum('duration_sec').alias('total_event_duration'),
        spark_sum('affected_customers').alias('total_affected'),
    )
    .withColumn('had_outage', lit(1))
)

print(f'Days with outage events: {outage_events.count():,}')

In [ ]:
# === Join sensor features with outage target ===
ml_features = (
    sensor_daily
    .join(
        outage_events.select('substation_id', col('event_date').alias('sensor_date'), 'had_outage', 'event_count', 'total_affected'),
        ['substation_id', 'sensor_date'],
        'left'
    )
    .withColumn('had_outage', when(col('had_outage').isNull(), 0).otherwise(1))
    .withColumn('day_of_week', dayofweek('sensor_date'))
    .withColumn('month', month('sensor_date'))
    .na.fill(0)
    .withColumn('feature_timestamp', current_timestamp())
)

ml_features.write.mode('overwrite').format('delta').saveAsTable('gold_ml_features')

outage_rate = ml_features.filter(col('had_outage') == 1).count() / ml_features.count() * 100
print(f'\nGold ML features written: {ml_features.count():,} rows')
print(f'Columns: {len(ml_features.columns)}')
print(f'Outage rate: {outage_rate:.1f}%')

In [ ]:
spark.sql('OPTIMIZE gold_ml_features')
print('Feature table optimized')